<h1>Анализ трендов YouTube Russia</h1>

<p>
Проект направлен на <b>исследование закономерностей популярности видео на YouTube</b> 
на основе открытого датасета: <a href="https://www.kaggle.com/datasets/datasnaek/youtube-new" target="_blank"><code>RUvideos.csv</code> (YouTube Trending Video Dataset)</a> 
за период <b>14.11.2017 — 14.06.2018</b> (около 40 тыс. записей).<br><br>
</p>

<p><b>Основные цели анализа:</b></p>
<ul>
  <li>Определить, какие категории контента чаще попадают в тренды.</li>
  <li>Исследовать влияние времени публикации и дня недели< на просмотры.</li>
  <li>Проверить, влияет ли количество тегов на популярность видео.</li>
  <li>Изучить взаимосвязь между просмотрами, лайками и комментариями.</li>
  <li>Провести статистические тесты (Mann–Whitney, корреляции для проверки гипотез.</li>
</ul>



In [ ]:
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime as dt
import math
import os
import seaborn as sns
from scipy import stats
import json
import re
import phik
from phik import report
from phik.report import plot_correlation_matrix
import calendar
import warnings
import statsmodels
import sklearn
import patsy

In [1]:
file_path = r'C:\Users\Станция\Desktop\Jupyter\Youtube\RUvideos.csv'

In [2]:
# файл не читался из-за проблем с кодировкой, добавляем encoding_errors='replace' чтобы не выпадала ошибка при считывании файла 
df = pd.read_csv(file_path, encoding='utf-8', encoding_errors='replace')

NameError: name 'pd' is not defined

In [ ]:
df.head(10)

В этом датасете закодированы категории видео. В приложении шел файл json с расшифровкой категорий, добавим столбец с понятным названием категорий.

In [ ]:
json_path = r'C:\Users\Станция\Desktop\Jupyter\Youtube\RU_category_id.json'

In [ ]:
# Загружаем JSON с категориями
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

In [ ]:
category_map = {
    int(item["id"]): item["snippet"]["title"]
    for item in data["items"]
    if "snippet" in item and "title" in item["snippet"]
}

In [ ]:
# Добавляем расшифровку категорий
df["category_name"] = df["category_id"].map(category_map)

In [ ]:
# проверяем итог
df.head(5)

In [ ]:
df.info()

In [ ]:
df.describe()

В среднем трендовые видео собирали по 240 тысяч просмотров, максимально почти 62 миллиона 800 тысяч

In [ ]:
df.duplicated().sum()

Всего 46 полных дубликатов на 40739 строк, удалим их

проверим дубликаты по id

In [ ]:
df = df.drop_duplicates()

In [ ]:
df.info()

In [ ]:
df[df.duplicated(subset='video_id')]

Получаем 560 дубликатов по id, есть названия "#NAME?" - это ошибка, видео от разных блогеров, сам номер говорит о том что id нет по той или иной причине.
Удалять такие строки не будем, зададим собственные id для этих видео.

In [ ]:
# фильтруем строки с некорректными id
mask_bad = df['video_id'] == '#NAME?'
count_bad = mask_bad.sum()

new_ids = [f"abcxyz{i+1}" for i in range(count_bad)]

df.loc[mask_bad, 'video_id'] = new_ids

Приведем тип данных в trending_date к datetime

In [ ]:
df['trending_date']= pd.to_datetime(df['trending_date'], format='%y.%d.%m')

In [ ]:
print(f'Мы имеет датасет за период с {df['trending_date'].min()} по {df['trending_date'].max()}')

In [ ]:
# топ-10 категорий по среднему числу просмотров
top_views = (
    df.groupby("category_name")["views"]
    .mean()
    .sort_values(ascending=False)
    .head(15)
    .reset_index()
)

plt.figure(figsize=(10,6))
sns.barplot(
    data=top_views,
    x="views",
    y="category_name",
    hue="category_name",   # ← теперь это столбец в DataFrame
    palette="viridis"
)
plt.title("Среднее количество просмотров по категориям")
plt.xlabel("Среднее количество просмотров")
plt.ylabel("Категория")
plt.tight_layout()
plt.show()

Топ 5 трендовых видео идут в категориях: Музыка, наука и технологии, развлечения, комедия, авто и транспорт

In [ ]:

top_categories = top_views["category_name"].tolist()

median_order = (
    df[df["category_name"].isin(top_categories)]
    .groupby("category_name")["views"]
    .median()
    .sort_values(ascending=False)
    .index.tolist()   # превращаем Index в обычный список
)

plt.figure(figsize=(10, 6))
sns.boxplot(
    data=df[df["category_name"].isin(top_categories)],
    x="category_name",
    y="views",
    hue="category_name",
    order=median_order,
    palette="viridis"
)
plt.yscale("log")
plt.title("Распределение просмотров по категориям (топ-10, отсортировано по медиане)")
plt.xlabel("Категория")
plt.ylabel("Просмотры (лог-шкала)")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
top_likes = (
    df.groupby("category_name")["likes"]
    .mean()
    .sort_values(ascending=False)
    .head(15)
    .reset_index()
)

plt.figure(figsize=(10,6))
sns.barplot(data = top_likes, x=top_likes.likes, y=top_likes.category_name, hue="category_name", palette="mako")
plt.title("Среднее количество лайков по категориям")
plt.xlabel("Среднее количество лайков")
plt.ylabel("Категория")
plt.show()

Музыка лидирует по лайкам, comedy с 4 места по просмотрам выходит на второе по лайкам, люди особо охотно лайкают комедийный контент. Далее лидеры идут как в предыдущем графике.

In [ ]:
df["like_ratio"] = df["likes"] / (df["likes"] + df["dislikes"])
sns.histplot(df["like_ratio"].dropna(), bins=30)
plt.title("Распределение доли лайков среди всех реакций")
plt.xlabel("Доля лайков")
plt.ylabel("Количество видео")
plt.show()

Большее количество видео имеет долю лайков от 80 процентов, в целом по графику видно что люди больше ставят лайки на видео контент нежели дислайки, в целом большая часть контента воспринимается позитивно

In [ ]:
mean_like_ratio = (
    df.groupby("category_name")["like_ratio"]
    .mean()
    .sort_values(ascending=False)
    .head(15)
    .reset_index()
)

# строим график
plt.figure(figsize=(10,6))
sns.barplot(
    data=mean_like_ratio,
    x="like_ratio",
    y="category_name",
    hue="category_name",
    palette="viridis",
    dodge=False,
    legend=False
)
plt.title("Средняя доля лайков среди всех реакций (топ-15 категорий)")
plt.xlabel("Средняя доля лайков (like_ratio)")
plt.ylabel("Категория")
plt.xlim(0, 1)  # ограничим ось X от 0 до 1
plt.tight_layout()
plt.show()

Самые "залайканые" видео это видео про милых животных, на втором месте "Мода и стиль", третье место - компьютерные игры. Четвертое место занимает жанр - топ 2 по просмотрам - Science&Technology. Лидер просмотров  - музыка, занимает 11 место по доле лайков, у исполнителей и коллективов много поклонников, но так же есть большие отряды "хейтеров".

In [ ]:
df["publish_time"] = pd.to_datetime(df["publish_time"], errors="coerce")
df["publish_hour"] = df["publish_time"].dt.hour

sns.barplot(x="publish_hour", y="views", data=df)
plt.title("Средние просмотры по часу публикации")
plt.xlabel("Час публикации (UTC)")
plt.ylabel("Среднее количество просмотров")
plt.show()

Самые высокие просмотры у видео которые выкладываются к 21-00. Далее идут 1 час ночи и 9 утра. Видео, выкладываемые в остальное время, имеют в среднем от 20 до 30 тысяч просмотров

In [ ]:
top_channels = (
    df.groupby("channel_title")[["views", "likes"]]
    .mean()
    .sort_values(by="views", ascending=False)
    .head(30)
    .reset_index()
)
top_channels

Топ три канала по просмотрам это: Kylie Jenner, YouTube Spotlight, FoxStarHindi. Любопытно у канала Kylie Jenner, нет информации по количеству лайков

In [ ]:
df[df['channel_title'] == 'Kylie Jenner']

Топ канал из нашего списка два дня попадал в тренды и набрал в среднем 28 млн просмотров в день

Все топ 30 каналов не русскоязычные, хочется посмотреть топ именно русскоязычных каналов. 

In [ ]:
def has_cyrillic(text):
    if isinstance(text, str):
        return bool(re.search(r'[А-Яа-яЁё]', text))
    return False

In [ ]:
# проверяем есть ли русские буквы в описании
df["is_russian"] = df["description"].apply(has_cyrillic)

df_ru = df[df["is_russian"] == True].copy()

print(f"Всего русскоязычных видео: {len(df_ru)} из {len(df)}")

In [ ]:
russian_channels = (
    df_ru.groupby("channel_title")[["views"]]
    .mean()
    .sort_values("views", ascending=False)
    .reset_index()
)

print(russian_channels.head(30))

Топ три русскоязычных канала по просмотрам: Comedy Radio, World Cup Free, ВОТ ТАК

In [ ]:
# Топ-10 категорий по среднему числу просмотров
top_views = (
    df_ru.groupby("category_name")["views"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

plt.figure(figsize=(10,6))
sns.barplot(data=top_views, x="views", y="category_name", hue="category_name", palette="viridis")
plt.title("Среднее количество просмотров по категориям")
plt.xlabel("Среднее количество просмотров")
plt.ylabel("Категория")
plt.show()

Топ 5 тот же что и с учетом англоязычных каналов, но позиции некоторых категорий не совпадают. Comedy и entertainment, занимают второе и третье место.

In [ ]:
top_likes = (
    df_ru.groupby("category_name")["likes"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

plt.figure(figsize=(10,6))
sns.barplot(data=top_likes, x="likes", y="category_name", hue="category_name", palette="mako")
plt.title("Среднее количество лайков по категориям")
plt.xlabel("Среднее количество лайков")
plt.ylabel("Категория")
plt.show()

Те же топ 5 что и с англоязычными кналами, но чаще лайкают науно-популярные видео чем comedy

In [ ]:
# выбираем наборы параметров для корреляции по phik
cols_for_phik = [
    "views", "likes", "dislikes", "comment_count",
    "like_ratio", "publish_hour", "category_name", "is_russian"
]

# фильтруем только нужные столбцы и убираем NaN
df_phik = df_ru[cols_for_phik].dropna().copy()


In [ ]:
phik_matrix = df_phik.phik_matrix(interval_cols=["views", "likes", "dislikes", "comment_count", "like_ratio", "publish_hour"])


In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(phik_matrix, cmap='coolwarm', annot=True, fmt=".2f", vmin=0, vmax=1)
plt.title("Phik корреляция между признаками русскоязычного YouTube")
plt.tight_layout()
plt.show()

Сильная связь между просмотрами и лайками. Так же между просмотрами и дислайками, чем больше дислайков тем больше просмотров, как бы это странно не звучало. Есть определенные пропорции лайков к дислайкам, чем больше лайков тем, соответственно, больше и дислайков. Так же чем больше лайков тем лучше продвигается видео.

Парные зависимости на графике, для проверки и подверждения корреляции.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12,10))

sns.scatterplot(data=df_ru, x="views", y="likes", ax=axes[0,0], alpha=0.3)
sns.scatterplot(data=df_ru, x="views", y="comment_count", ax=axes[0,1], alpha=0.3)
sns.scatterplot(data=df_ru, x="views", y="dislikes", ax=axes[1,0], alpha=0.3)
sns.scatterplot(data=df_ru, x="likes", y="dislikes", ax=axes[1,1], alpha=0.3)

for ax in axes.flat:
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.grid(True, linestyle="--", alpha=0.5)

plt.suptitle("Парные зависимости между ключевыми метриками", fontsize=14)
plt.tight_layout()
plt.show()

Линейная связь подтверждается, между лайками и просмотрами самое узкое облако разбросанных значений, и имеет близкий к 45 градусам наклон. Сильная зависимость видна и на графике. Умеренная связь между лайками и дислайками видна и на графике рассеивания, хотя  само облако более широкое и наклон дальше от идеальных 45 градусов чем левая-верхняя картинка.

In [ ]:
# сколько уникальных видео было в трендах каждый день
daily_trend_count = df_ru.groupby('trending_date')['video_id'].nunique().reset_index(name='videos_in_trend')

plt.figure(figsize=(10,5))
sns.lineplot(data=daily_trend_count, x='trending_date', y='videos_in_trend', color='teal')
plt.title('Количество трендовых видео по дням')
plt.xlabel('Дата')
plt.ylabel('Уникальных видео в трендах')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Среднее количество видео в день: {daily_trend_count['videos_in_trend'].mean():.1f}")
print(f"Минимум: {daily_trend_count['videos_in_trend'].min()} | Максимум: {daily_trend_count['videos_in_trend'].max()}")

Пик трендовых видео был в середине марта 2018, самый низкий уровень был в середине мая 2018г. Среднее количество тредовых видео находится в районе 170 в день.

In [ ]:
# считаем уникальные даты тренда на одно видео
trend_days = df_ru.groupby('video_id')['trending_date'].nunique().reset_index(name='days_in_trend')

print("📊 Средние показатели по дням в тренде:")
print(trend_days['days_in_trend'].describe())

plt.figure(figsize=(8,5))
sns.boxplot(x=trend_days['days_in_trend'], color='orange')
plt.title('Распределение продолжительности нахождения видео в трендах (в днях)')
plt.xlabel('Количество дней в тренде')
plt.grid(alpha=0.3)
plt.show()

print(f"Среднее: {trend_days['days_in_trend'].mean():.1f} дней")
print(f"Медиана: {trend_days['days_in_trend'].median():.1f}")
print(f"Максимум: {trend_days['days_in_trend'].max()} дней")

Среднее и медианное время нахождения видео в трендах 1 день. Максимально одно видео пробыло в трендах 4 дня.

In [ ]:
# получаем id этого видео
top_video_id = trend_days.loc[trend_days['days_in_trend'] == trend_days['days_in_trend'].max(), 'video_id'].values[0]

# извлекаем всю информацию о нём
top_video_info = df_ru[df_ru['video_id'] == top_video_id]

top_video_info

Это видео BadComedian, жанр comedy, обзор кинофильмов.

In [ ]:
channel_trend_count = (
    df_ru.groupby('channel_title')['video_id']
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name='trending_videos')
)

top_channels_count = channel_trend_count.head(15)

plt.figure(figsize=(10,6))
sns.barplot(data=top_channels_count, x='trending_videos', y='channel_title', hue='channel_title', palette='mako')
plt.title('Каналы с наибольшим числом видео в трендах')
plt.xlabel('Количество уникальных трендовых видео')
plt.ylabel('Канал')
plt.tight_layout()
plt.show()

top_channels_count.head(10)


Топ три канала-генератора трендовых видео это Эхо Москвы, Анатолий Шарий и Модные Практики с Паукште Ириной Михайловной. 

In [ ]:
plt.figure(figsize=(12,6))
sns.boxplot(data=df_ru, x='category_name', y='likes')
plt.xticks(rotation=90)
plt.title('Распределение лайков по категориям')
plt.tight_layout()
plt.show()

plt.figure(figsize=(12,6))
sns.boxplot(data=df_ru, x='category_name', y='dislikes')
plt.xticks(rotation=90)
plt.title('Распределение дизлайков по категориям')
plt.tight_layout()
plt.show()

Без логарифмической шкалы данные выглядят плохо и непонятно, нужно добавить лог. шкалу, заодно отсортируем данные по медиане

In [ ]:
# Сортировка категорий по медиане лайков 
order_likes = (
    df_ru.groupby("category_name")["likes"]
    .median()
    .sort_values(ascending=False)
    .index
)


plt.figure(figsize=(12,6))
sns.boxplot(
    data=df_ru,
    x="category_name",
    y="likes",
    order=order_likes,
    hue="category_name",     
    palette="mako",
    legend=False
)
plt.yscale("log")
plt.xticks(rotation=90)
plt.title("Распределение лайков по категориям (лог-шкала, отсортировано по медиане)")
plt.tight_layout()
plt.show()


# Сортировка категорий по медиане дизлайков
order_dislikes = (
    df_ru.groupby("category_name")["dislikes"]
    .median()
    .sort_values(ascending=False)
    .index
)


plt.figure(figsize=(12,6))
sns.boxplot(
    data=df_ru,
    x="category_name",
    y="dislikes",
    order=order_dislikes,
    hue="category_name",
    palette="rocket",
    legend=False
)
plt.yscale("log")
plt.xticks(rotation=90)
plt.title("Распределение дизлайков по категориям (лог-шкала, отсортировано по медиане)")
plt.tight_layout()
plt.show()

Видим трех лидеров по медианным лайкам: Science&Tecnology, Autos&Vehicles, Entertainment
Три лидера по медианным дислайкам: Music, Science&Tecnology, Autos&Vehicles. 
Есть интересный момент, катагория Movies, показывет самый низкий разброс данных, самые стабильные данные. Либо по этой категории очень мало наблюдений?

In [ ]:
category_counts = (
    df_ru["category_name"]
    .value_counts()
    .reset_index()
)

category_counts

Только одно видео категории Movies представленно, этим объясняется "низкий разброс" данных

In [ ]:
# День недели: число (0=Mon … 6=Sun) и удобная подпись
df_ru['weekday_num'] = df_ru['trending_date'].dt.weekday
eng_days = list(calendar.day_name)  # ['Monday', ..., 'Sunday']
rus_days = ['Понедельник','Вторник','Среда','Четверг','Пятница','Суббота','Воскресенье']
map_rus = dict(zip(range(7), rus_days))
df_ru['weekday'] = df_ru['weekday_num'].map(map_rus)

#  Агрегаты: средние просмотры и количество видео по дням недели
order = list(range(7))  # порядок Пн→Вс
views_by_weekday = (
    df_ru.groupby('weekday_num', as_index=False)
         .agg(mean_views=('views','mean'),
              count_videos=('video_id','nunique'))
         .sort_values('weekday_num')
)
views_by_weekday['weekday'] = views_by_weekday['weekday_num'].map(map_rus)

#  ГРАФИК 1: Средние просмотры по дням недели 
plt.figure(figsize=(10,6))
plt.bar(views_by_weekday['weekday'], views_by_weekday['mean_views'])
plt.title('Средние просмотры трендовых видео по дням недели')
plt.xlabel('День недели')
plt.ylabel('Средние просмотры')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


#  ГРАФИК 2: Boxplot просмотров по дням недели (лог-шкала) 
groups = [df_ru.loc[df_ru['weekday_num']==w, 'views'].values for w in order]

plt.figure(figsize=(10,6))
bp = plt.boxplot(groups, labels=[map_rus[w] for w in order], showfliers=False)  
plt.yscale('log')  
plt.title('Распределение просмотров по дням недели (лог-шкала, без выбросов)')
plt.xlabel('День недели')
plt.ylabel('Просмотры (лог-шкала)')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

Средние просмотры по дням недели сильно не отличаются. Средние просмотры в пятницу, субботу и воскресенье, выглядят выше чем остальные дни, но не намного. Самый низкий уровень средних просмотров во вторник, так же во вторник самый низкий разброс значений.

Проверим несколько гипотез на базе нашего датасета.
Первая гипотеза о равенстве/различии просмотров в выходные и будни
Вторая гипотеза о равенстве/различии промотров между топ жанром Music и остальными жанрами

In [ ]:
# отделим выходной день от буднего
df_ru['is_weekend'] = df_ru['trending_date'].dt.weekday >= 5

# разделим на 2 выборки
weekend_views = df_ru.loc[df_ru['is_weekend'], 'views']
weekday_views = df_ru.loc[~df_ru['is_weekend'], 'views']

# логарифмируем, чтобы убрать сильный перекос
weekend_log = np.log1p(weekend_views)
weekday_log = np.log1p(weekday_views)

In [ ]:
# Тест Шапиро-Уилка для нормальности
print("Shapiro weekday:", stats.shapiro(weekday_log.sample(5000)))
print("Shapiro weekend:", stats.shapiro(weekend_log.sample(5000)))

In [ ]:
# Тест Манна–Уитни (альтернатива: у Music просмотров больше)
u_stat, p_value = stats.mannwhitneyu(weekend_views, weekday_views, alternative='two-sided')

print(f"p-value = {p_value:.6f}")

if p_value < 0.05:
    print("Отвергаем H0")
else:
    print("Различия по просмотрам между Weekend и рабочими нями недели статистически незначимы.")

In [ ]:
# Выделяем просмотры для категории Music и для остальных
music_views = df_ru.loc[df_ru['category_name'] == 'Music', 'views']
other_views = df_ru.loc[df_ru['category_name'] != 'Music', 'views']

# Тест Шапиро-Уилка для нормальности
print("Shapiro music:", stats.shapiro(music_views.sample(1400)))
print("Shapiro other:", stats.shapiro(other_views.sample(1400)))

Очень низкий p-value показывает тест Шапиро-Уилка, значит распределения не нормальны. Будем использовать тест Манна-Уитни

H0: Средние просмотры в категории Music не отличаются от других категорий.

H1: Средние просмотры в Music выше.

In [ ]:
# Тест Манна–Уитни (альтернатива: у Music просмотров больше)
u_stat, p_value = stats.mannwhitneyu(music_views, other_views, alternative='greater')

print(f"p-value = {p_value:.100f}")

if p_value < 0.05:
    print("Отвергаем H0: у категории 'Music' просмотров больше, чем у остальных.")
else:
    print("Различия по просмотрам между 'Music' и другими категориями статистически незначимы.")

Проверим гипотезу о том что чем больше тегов - тем больше просмотров.
Н0 - просмотры у видео с небольшим и с большим количеством тегов равны
Н1 - больше просмотров у видео с большим количеством тегов

In [ ]:
# Сосчитаем теги
def tag_count(s):
    if not isinstance(s, str) or s.strip()=='' or s.strip().lower()=='[none]':
        return 0
    # теги в датасете разделены символом '|'
    return s.count('|') + 1

df_work = df_ru.copy()
df_work['tags_count'] = df_work['tags'].apply(tag_count)

print(df_work['tags_count'].describe())
print("Пример распределения:\n", df_work['tags_count'].value_counts().sort_index().head(20))

Для начала применим корреляцию Спирмена (данные будут не нормально распределены, много одинаковых значений)

In [ ]:
#корреляция Спирмена (монотонная связь tags_count ~ views) ===
mask = df_work['views'].notna() & df_work['tags_count'].notna()
rho, p = stats.spearmanr(df_work.loc[mask, 'tags_count'], df_work.loc[mask, 'views'])
if p < 0.05:
    print(f"Spearman rho = {rho:.3f}, p-value = {p:.3g}, Нулевая гипотеза отвергается")
else:
    print(f"Spearman rho = {rho:.3f}, p-value = {p:.3g}, Нулевая гипотеза не может быть отвергнута")

In [ ]:
# Манн–Уитни: много тегов vs мало ===
# делим порог по медиане числа тегов, чтобы получить две группы схожего размера
medi_tags = df_work['tags_count'].median()
low  = df_work.loc[df_work['tags_count'] <=  medi_tags, 'views']
high = df_work.loc[df_work['tags_count'] >   medi_tags, 'views']

u, p = stats.mannwhitneyu(high, low, alternative='greater')  # H1: у "много тегов" просмотров больше
if p < 0.05:
    print(f"p-value={p:.3g}, Количество наблюдений low={len(low)}, Количество наблюдений high={len(high)}, Нулевая гипотеза отвергается")
else:
    print(f"p-value={p:.3g}, Количество наблюдений low={len(low)}, Количество наблюдений high={len(high)}, Нулевая гипотеза не может быть отвергнута")

#визуализация: бокс-плот в лог-шкале
plt.figure(figsize=(6,5))
sns.boxplot(data=pd.DataFrame({'views': pd.concat([low, high], ignore_index=True),
                               'group': ['≤median tags']*len(low)+['>median tags']*len(high)}),
            x='group', y='views')
plt.yscale('log')
plt.title('Просмотры vs количество тегов (по медиане)')
plt.xlabel('Группа по числу тегов')
plt.ylabel('Просмотры (лог-шкала)')
plt.tight_layout()
plt.show()

Статистические тесты и boxplot показывают нам небольшое, но все же статистически значимое превышение просмотров у видео с количеством тегов больше медианного.

<h1>Итоги проекта YouTube Trending </h1>

<h2>Данные и период</h2>
<p>
Проанализирован датасет <b>трендов YouTube Russia</b> за <b>14.11.2017 — 14.06.2018</b> (~40 тыс. строк после очистки).<br>
Категории расшифрованы из JSON; приведены даты/времена, обработаны дубликаты и технические артефакты (<code>#NAME?</code>), 
добавлены производные признаки (час публикации, like-ratio, фильтр русскоязычных видео и т. п.).
</p>

<h2>Характер данных</h2>
<p>
Распределения метрик <b>сильно перекошены</b> (длинные хвосты), поэтому для корректного анализа использовались 
<b>логарифмическая шкала</b> и <b>непараметрические тесты</b>.
</p>

<h2>Корреляции</h2>
<ul>
<li>Наблюдается <b>сильная положительная связь</b> <code>views ↔ likes</code> и <b>умеренная</b> <code>views ↔ comment_count</code> — чем выше охват, тем выше вовлечённость.</li>
<li>Корреляции с категориями присутствуют, но выражены слабее.</li>
</ul>

<h2>Категории</h2>
<ul>
<li><b>Music</b> — уверенный лидер по просмотрам (и по средним, и по медианам).</li>
<li><b>Entertainment</b> и <b>Comedy</b> — следующий эшелон.</li>
<li><b>Science & Technology</b> — стабильные средние показатели.</li>
<li><b>Howto & Style / People & Blogs / Travel & Events / Sports</b> — существенно ниже по охватам.</li>
</ul>

<p><b>Статистически подтверждено (Mann–Whitney):</b> у категории <b>Music</b> просмотров <b>значимо больше</b>, чем у остальных.</p>

<h2>Время и календарь</h2>
<ul>
<li>Эффект <b>дня недели</b> статистически незначим (<code>p ≈ 0.14</code>). Наблюдаемый косметический рост к выходным не подтверждён тестом.</li>
<li><b>Час публикации</b> в факторной модели — пограничный/незначимый.</li>
</ul>

<h2>Теги</h2>
<ul>
<li>Есть <b>слабая, но стабильная положительная связь</b> между количеством тегов и просмотрами.</li>
<li>По тесту <b>Манна–Уитни</b> у видео с числом тегов <b>выше медианы просмотры статистически выше</b>.</li>
<li><b>Вывод:</b> теги помогают как <b>тонкая настройка</b>, а не как основной драйвер охватов.</li>
</ul>

<h2>Ограничения</h2>
<ul>
<li>Данные только по <b>трендовым</b> видео (selection bias).</li>
<li>Период ограничен <b>2017–2018</b> годами, регион <b>RU</b>.</li>
<li>Нет поведенческих метрик (<b>CTR, удержание, скорость набора просмотров</b>).</li>
<li>Нет «контрольной» выборки <b>обычных видео</b>, поэтому вероятность попадания в тренды оценивалась через прокси <b>«долго / недолго держится»</b>.</li>
</ul>

<h2>Ключевые выводы для продукта и контента</h2>

<h3>1) Жанр — главный фактор успеха</h3>
<p>Музыка и развлекательный контент системно выигрывают по охватам и длительности нахождения в трендах.</p>

<h3>2) Календарь почти не влияет</h3>
<p>Ставка на «пятницу/выходные» статистически не оправдана; важнее сама <b>идея видео, эмоции и вовлечённость</b>.</p>

<h3>3) Метаданные — доводка, не двигатель</h3>
<p>Релевантные теги и информативное описание немного помогают; <b>длинные заголовки — минус</b>.</p>

<h3>4) Фокус на вовлечении</h3>
<p>Для существенного улучшения прогнозов нужны <b>поведенческие метрики</b> (CTR, удержание первых секунд, скорость набора просмотров) и <b>данные о размере канала</b>.</p>

<hr>

<h2>Общий вывод</h2>
<p>
На YouTube RU (2017–2018) главным фактором популярности является <b>жанр контента</b>, 
в меньшей степени — метаданные и расписание публикации.<br>
